# H2a / H2b — Populism and accusations of lying

**H2a**: populist politicians are more likely to *accuse* opponents of lying.  
**H2b**: populist politicians are more likely to *be accused* of lying.

Design: speaker-year panel (see `lib/panel.py`). Negative binomial count models
with `offset(log(n_sentences))`:

- **Accuser side** (H2a): DV = `n_accusations_made`.
- **Accusee side** (H2b): DV = `n_accusations_received`. *Caveat*: received
  counts exist only where targets resolved (~50%, dataset-dependent) \u2192 dataset FE
  mandatory + robustness on high-resolution datasets.

Populism = V-Party `v2xpa_popul` (0\u20131) of the speaker's party that year.
Controls: gender, age, education, left_right, in_cabinet, country + year FE.

> This notebook is the **template** for all accuser/accusee-side hypotheses \u2014
> H3a\u2013c and H4a\u2013d swap the focal regressor, same skeleton.

In [ ]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from lib import panel, viz
viz.apply_style()

In [ ]:
df = panel.load_panel()
print(df.shape)

# Estimation sample: party-level vars present + enough speech to have exposure
est = df.dropna(subset=["populism", "left_right", "female", "age"]).copy()
est = est[est["n_sentences"] >= 50]
est["edu"] = est["highest_isced"]          # 5\u20138 ordinal
print(f"estimation sample: {len(est):,} speaker-years, "
      f"{est['speaker_id'].nunique():,} speakers, {est['country'].nunique()} countries")

## Descriptive: accusation rates by populism quartile

In [ ]:
est["pop_q"] = pd.qcut(est["populism"], 4, labels=["Q1 low", "Q2", "Q3", "Q4 high"])
desc = (est.groupby("pop_q", observed=True)
           .apply(lambda g: pd.Series({
               "made_per_10k": g["n_accusations_made"].sum() / g["n_sentences"].sum() * 1e4,
               "received_per_10k": g["n_accusations_received"].sum() / g["n_sentences"].sum() * 1e4,
           }), include_groups=False))

fig, ax = plt.subplots()
desc.plot.bar(ax=ax, rot=0)
ax.set_ylabel("accusations per 10k sentences")
ax.set_title("Accusations made / received by party-populism quartile")
viz.savefig(fig, "h2_descriptive_quartiles")
plt.show()

## H2a \u2014 accuser side

In [ ]:
FORMULA_MADE = ("n_accusations_made ~ populism + left_right + female + age + edu "
                "+ in_cabinet + C(country) + C(year)")

m_made = smf.glm(FORMULA_MADE, data=est,
                 family=sm.families.NegativeBinomial(alpha=1.0),
                 offset=est["log_exposure"])\
            .fit(cov_type="cluster", cov_kwds={"groups": est["speaker_id"]})

focal = ["populism", "left_right", "female", "age", "edu", "in_cabinet"]
print(m_made.summary2().tables[1].loc[focal].round(4))
irr = np.exp(m_made.params["populism"])
print(f"\nIRR populism (0\u21921): {irr:.2f} \u2014 a maximally populist party's MPs accuse "
      f"{(irr-1)*100:+.0f}% more per sentence spoken (H2a: expect > 1)")

## H2b \u2014 accusee side (dataset FE, resolution caveat)

In [ ]:
FORMULA_RECV = ("n_accusations_received ~ populism + left_right + female + age + edu "
                "+ in_cabinet + C(source_dataset) + C(year)")

m_recv = smf.glm(FORMULA_RECV, data=est,
                 family=sm.families.NegativeBinomial(alpha=1.0),
                 offset=est["log_exposure"])\
            .fit(cov_type="cluster", cov_kwds={"groups": est["speaker_id"]})

print(m_recv.summary2().tables[1].loc[focal].round(4))
irr = np.exp(m_recv.params["populism"])
print(f"\nIRR populism (0\u21921): {irr:.2f} (H2b: expect > 1)")

## Coefficient plot (both sides)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for i, (m, lab) in enumerate([(m_made, "made (H2a)"), (m_recv, "received (H2b)")]):
    est_ = m.params[focal]; ci = m.conf_int().loc[focal]
    y = np.arange(len(focal)) + (i - 0.5) * 0.25
    ax.errorbar(est_, y, xerr=[est_ - ci[0], ci[1] - est_],
                fmt="o", capsize=3, label=lab)
ax.axvline(0, color="grey", lw=1)
ax.set_yticks(range(len(focal))); ax.set_yticklabels(focal)
ax.set_xlabel("coefficient (log-rate)")
ax.set_title("Drivers of accusations of lying \u2014 speaker-year models")
ax.legend()
viz.savefig(fig, "h2_coefplot")
plt.show()

TODO robustness:
- estimate NB `alpha` instead of fixing at 1 (or Poisson + robust SE comparison);
- H2b restricted to datasets with target-resolution \u2265 40%;
- binary populist-party indicator (e.g. `populism > 0.5`) instead of continuous;
- interaction `populism \u00d7 left_right` (right-wing-populism concentration \u2192 feeds H4).